In [3]:
import os, urllib.request
if not os.path.exists("utils.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/abgoswam/swe_in_prod_vizuara_01/main/utils.py",
        "utils.py"
    )

import json, random
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from utils import (TARGET, MOCK_FILE, MockEnv, SYSTEM, NO_COMMAND, first_bash_block, generate, scripted_sampler, show_rollout, show_group)

for _opt in ["display.max_colwidth", "display.max_rows", "display.max_columns", "display.width"]:
    pd.set_option(_opt, None)

random.seed(0); torch.manual_seed(0)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device: ", DEV)

device:  cuda


## Load one instance

In [4]:
from datasets import load_dataset

ds = load_dataset("princeton-nlp/SWE-bench_Verified", split="test") 
print(ds)

# A small single-file task keeps the walkthrough readable.
cands = [i for i, r in enumerate(ds)
            if r["patch"].count("diff --git") == 1 and len(r["patch"]) < 1800]

inst = ds[cands[0]] 
fail_to_pass = json.loads(inst["FAIL_TO_PASS"])
pass_to_pass = json.loads(inst["PASS_TO_PASS"])

display(pd.DataFrame(
    [(k, inst.get(k)) for k in ["instance_id", "repo", "base_commit", "version", "difficulty"]],
    columns=["field", "value"],
))

display(pd.DataFrame([
    ("1.1", "problem_statement", "the agent",  "the input: a raw GitHub issue"),
    ("-",   "patch",             "nobody",     "reference solution; unused in this notebook"),
    ("1.2", "test_patch",        "the grader", "adds the tests that define 'fixed'"),
    ("1.3", "FAIL_TO_PASS",      "the grader", "must go red -> green"),
    ("1.4", "PASS_TO_PASS",      "the grader", "must stay green"),
], columns=["section", "field", "who sees it", "job"]))

README.md:   0%|          | 0.00/3.34k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.10MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['repo', 'instance_id', 'base_commit', 'patch', 'test_patch', 'problem_statement', 'hints_text', 'created_at', 'version', 'FAIL_TO_PASS', 'PASS_TO_PASS', 'environment_setup_commit', 'difficulty'],
    num_rows: 500
})


,field,value
0,instance_id,astropy__astropy-12907
1,repo,astropy/astropy
2,base_commit,d16bfe05a744909de4b27f5875fe0d4ed41ce607
3,version,4.3
4,difficulty,15 min - 1 hour


,section,field,who sees it,job
0,1.1,problem_statement,the agent,the input: a raw GitHub issue
1,-,patch,nobody,reference solution; unused in this notebook
2,1.2,test_patch,the grader,adds the tests that define 'fixed'
3,1.3,FAIL_TO_PASS,the grader,must go red -> green
4,1.4,PASS_TO_PASS,the grader,must stay green


## 1.1 problem_statement — the agent's entire input
#### A raw GitHub issue. The agent is never told which file to open, or that separable.py exists. Finding it — localization — is most of the real difficulty of SWE-bench.

In [5]:
print(inst["problem_statement"])

Modeling's `separability_matrix` does not compute separability correctly for nested CompoundModels
Consider the following model:

```python
from astropy.modeling import models as m
from astropy.modeling.separable import separability_matrix

cm = m.Linear1D(10) & m.Linear1D(5)
```

It's separability matrix as you might expect is a diagonal:

```python
>>> separability_matrix(cm)
array([[ True, False],
       [False,  True]])
```

If I make the model more complex:
```python
>>> separability_matrix(m.Pix2Sky_TAN() & m.Linear1D(10) & m.Linear1D(5))
array([[ True,  True, False, False],
       [ True,  True, False, False],
       [False, False,  True, False],
       [False, False, False,  True]])
```

The output matrix is again, as expected, the outputs and inputs to the linear models are separable and independent of each other.

If however, I nest these compound models:
```python
>>> separability_matrix(m.Pix2Sky_TAN() & cm)
array([[ True,  True, False, False],
       [ True,  True, False, 

In [6]:
print(inst["test_patch"])

diff --git a/astropy/modeling/tests/test_separable.py b/astropy/modeling/tests/test_separable.py
--- a/astropy/modeling/tests/test_separable.py
+++ b/astropy/modeling/tests/test_separable.py
@@ -28,6 +28,13 @@
 p1 = models.Polynomial1D(1, name='p1')
 
 
+cm_4d_expected = (np.array([False, False, True, True]),
+                  np.array([[True,  True,  False, False],
+                            [True,  True,  False, False],
+                            [False, False, True,  False],
+                            [False, False, False, True]]))
+
+
 compound_models = {
     'cm1': (map3 & sh1 | rot & sh1 | sh1 & sh2 & sh1,
             (np.array([False, False, True]),
@@ -52,7 +59,17 @@
     'cm7': (map2 | p2 & sh1,
             (np.array([False, True]),
              np.array([[True, False], [False, True]]))
-            )
+            ),
+    'cm8': (rot & (sh1 & sh2), cm_4d_expected),
+    'cm9': (rot & sh1 & sh2, cm_4d_expected),
+    'cm10': ((rot & sh1) & sh2, cm_4d_expected),
+    

In [11]:
display(pd.DataFrame([
    ("existing cases",       6, "PASS_TO_PASS"),
    ("new, already green",   2, "PASS_TO_PASS"),
    ("new, red until fixed", 2, "FAIL_TO_PASS"),
    ("other tests in the file", 5, "PASS_TO_PASS"),
], columns=["group", "n_tests", "list"]))

print(f"\nFAIL_TO_PASS {len(fail_to_pass)}   PASS_TO_PASS {len(pass_to_pass)}")

,group,n_tests,list
0,existing cases,6,PASS_TO_PASS
1,"new, already green",2,PASS_TO_PASS
2,"new, red until fixed",2,FAIL_TO_PASS
3,other tests in the file,5,PASS_TO_PASS



FAIL_TO_PASS 2   PASS_TO_PASS 13


## FAIL_TO_PASS - must go red -> green
#### The half of the reward that says you solved the issue.

In [10]:
count = 0
for t in fail_to_pass:
    count += 1
    print(t)
print("Total failed", count)

astropy/modeling/tests/test_separable.py::test_separable[compound_model6-result6]
astropy/modeling/tests/test_separable.py::test_separable[compound_model9-result9]
Total failed 2


## PASS_TO_PASS - must stay green

In [9]:
count = 0
for t in pass_to_pass:
    count += 1
    print(t)
print("Total passed:", count)

astropy/modeling/tests/test_separable.py::test_coord_matrix
astropy/modeling/tests/test_separable.py::test_cdot
astropy/modeling/tests/test_separable.py::test_cstack
astropy/modeling/tests/test_separable.py::test_arith_oper
astropy/modeling/tests/test_separable.py::test_separable[compound_model0-result0]
astropy/modeling/tests/test_separable.py::test_separable[compound_model1-result1]
astropy/modeling/tests/test_separable.py::test_separable[compound_model2-result2]
astropy/modeling/tests/test_separable.py::test_separable[compound_model3-result3]
astropy/modeling/tests/test_separable.py::test_separable[compound_model4-result4]
astropy/modeling/tests/test_separable.py::test_separable[compound_model5-result5]
astropy/modeling/tests/test_separable.py::test_separable[compound_model7-result7]
astropy/modeling/tests/test_separable.py::test_separable[compound_model8-result8]
astropy/modeling/tests/test_separable.py::test_custom_model_separable
Total passed: 13


## 2. The Environment

### 2.1 Initial Stat
#### One file, 29 lines — the real _cstack from separable.py at base_commit. The bug is the asymmetry: cleft is assigned left, but cright is assigned 1.

In [12]:
env = MockEnv(fail_to_pass)

print("state = files in the environment:", list(env.fs), "\n")
print(env.fs[TARGET])   

state = files in the environment: ['astropy/modeling/separable.py'] 

def _cstack(left, right):
    """
    Function corresponding to '&' operation.

    Parameters
    ----------
    left, right : `astropy.modeling.Model` or ndarray
        If input is of an array, it is the output of `coord_matrix`.

    Returns
    -------
    result : ndarray
        Result from this operation.

    """
    noutp = _compute_n_outputs(left, right)

    if isinstance(left, Model):
        cleft = _coord_matrix(left, 'left', noutp)
    else:
        cleft = np.zeros((noutp, left.shape[1]))
        cleft[: left.shape[0], : left.shape[1]] = left
    if isinstance(right, Model):
        cright = _coord_matrix(right, 'right', noutp)
    else:
        cright = np.zeros((noutp, right.shape[1]))
        cright[-right.shape[0]:, -right.shape[1]:] = 1

    return np.hstack([cleft, cright])


### 2.2 Take an Acion

### An action is one bash command. The first three are read-only — they return an observation and leave the state alone. Only the last one, a write, moves it.

In [13]:
FIXED = MOCK_FILE.replace("= 1", "= right")     # the one-token fix

rows = []
for action in ["ls",
               f'grep -n "cright" {TARGET}',
               "python -m pytest",
               f"cat > {TARGET} <<'EOF'\n{FIXED}\nEOF"]:
    before = dict(env.fs)
    obs = env.run(action)
    rows.append({"action":        action.splitlines()[0] + (" ..." if "\n" in action else ""),
                 "state changed": env.fs != before,
                 "observation":   obs.replace("\n", " | ")[:60]})

display(pd.DataFrame(rows))

,action,state changed,observation
0,ls,False,astropy/modeling/separable.py | setup.py | README.rst | test
1,"grep -n ""cright"" astropy/modeling/separable.py",False,"24: cright = _coord_matrix(right, 'right', noutp) | 2"
2,python -m pytest,False,FAILED astropy/modeling/tests/test_separable.py::test_separa
3,cat > astropy/modeling/separable.py <<'EOF' ...,True,[mock env] wrote 29 lines to astropy/modeling/separable.py


### 2.3 New state
#### The write landed. Here is the state now — and its difference from the initial state is the candidate patch. Nothing else builds it.

In [14]:
print("new state:\n")   
print(env.fs[TARGET])   

print("\ninitial state vs new state - the candidate patch: \n")
print(env.patch())

new state:

def _cstack(left, right):
    """
    Function corresponding to '&' operation.

    Parameters
    ----------
    left, right : `astropy.modeling.Model` or ndarray
        If input is of an array, it is the output of `coord_matrix`.

    Returns
    -------
    result : ndarray
        Result from this operation.

    """
    noutp = _compute_n_outputs(left, right)

    if isinstance(left, Model):
        cleft = _coord_matrix(left, 'left', noutp)
    else:
        cleft = np.zeros((noutp, left.shape[1]))
        cleft[: left.shape[0], : left.shape[1]] = left
    if isinstance(right, Model):
        cright = _coord_matrix(right, 'right', noutp)
    else:
        cright = np.zeros((noutp, right.shape[1]))
        cright[-right.shape[0]:, -right.shape[1]:] = right

    return np.hstack([cleft, cright])

initial state vs new state - the candidate patch: 

--- a/astropy/modeling/separable.py
+++ b/astropy/modeling/separable.py
@@ -24,6 +24,6 @@
         cright = _coord_matrix(r

## 3. The Agent

### 3.1 The model
#### Qwen2.5-Coder-0.5B-Instruct — the policy.

In [17]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16 if DEV == "cuda" else torch.float32,
    attn_implementation="sdpa").to(DEV)

print(f"{MODEL}\n{sum(p.numel() for p in model.parameters()):,} parameters, none trainable yet")

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen/Qwen2.5-Coder-0.5B-Instruct
494,032,768 parameters, none trainable yet


### 3.2 The Harness

In [19]:
# run_agent() drives the model through the harness. One call returns one
# rollout: the whole context, since in RL the trajectory is the training example.

def run_agent(model, tok, max_turns=4, temperature=1.0, sample=generate):
    env = MockEnv(fail_to_pass)
    context = [{"role": "system", "content": SYSTEM},
               {"role": "user",   "content": f"ISSUE:\n{inst['problem_statement'][:1500]}"}]

    for _ in range(max_turns):
        prompt = tok.apply_chat_template(context, tokenize=False, add_generation_prompt=True)
        reply  = sample(model, tok, prompt, temperature=temperature)
        action = first_bash_block(reply)
        obs    = env.run(action) if action else NO_COMMAND
        context += [{"role": "assistant", "content": reply},
                    {"role": "user",      "content": obs[:800]}]

    return dict(messages=context, patch=env.patch(), final=dict(env.fs), calls=env.calls)

In [20]:
rollout = run_agent(model, tok)

In [21]:
show_rollout(rollout)

10 messages = 2 setup + 2 per turn x 4 turns



,i,turn,role,written by,chars,preview
0,0,setup,system,harness,505,You are a software engineering agent. You are in a Python repository. | Fix the bug described in the issue. | | Respond with exactly ONE bash command per messa […]
1,1,setup,user,harness,1253,ISSUE: | Modeling's `separability_matrix` does not compute separability correctly for nested CompoundModels | Consider the following model:\r | \r | ```python\r | […]
2,2,0,assistant,model,75,```bash | ls | grep -n 'astropy.modeling.separable.py' | awk '{print $1}' | ```
3,3,0,user,environment,56,astropy/modeling/separable.py | setup.py | README.rst | tests/
4,4,1,assistant,model,35,```bash | python setup.py install | ```
5,5,1,user,environment,101,"bash: python: command not found | [mock env] implements only: ls, cat, grep, pytest, cat > FILE <<'EOF'"
6,6,2,assistant,model,36,```bash | python -m unittest tests | ```
7,7,2,user,environment,101,"bash: python: command not found | [mock env] implements only: ls, cat, grep, pytest, cat > FILE <<'EOF'"
8,8,3,assistant,model,34,```bash | python -m pytest tests | ```
9,9,3,user,environment,251,FAILED astropy/modeling/tests/test_separable.py::test_separable[compound_model6-result6] | FAILED astropy/modeling/tests/test_separable.py::test_separable[compo […]



commands:
  $ ls | grep -n 'astropy.modeling.separable.py' | awk '{print $1}'
  $ python setup.py install
  $ python -m unittest tests
  $ python -m pytest tests

candidate patch:
(no file changed)


### 3.3.2 With an Oracle model

In [22]:
SCRIPT = [
    f"Let me read the file.\n```bash\ncat {TARGET}\n```",
    f"The cright branch assigns 1 instead of right. Fixing it.\n"
    f"```bash\ncat > {TARGET} <<'EOF'\n{FIXED}\nEOF\n```",
    "Now run the tests.\n```bash\npython -m pytest\n```",
    "Check the tree.\n```bash\nls\n```",
]

expert = run_agent(model, tok, sample=scripted_sampler(SCRIPT))

In [23]:
show_rollout(expert)

10 messages = 2 setup + 2 per turn x 4 turns



,i,turn,role,written by,chars,preview
0,0,setup,system,harness,505,You are a software engineering agent. You are in a Python repository. | Fix the bug described in the issue. | | Respond with exactly ONE bash command per messa […]
1,1,setup,user,harness,1253,ISSUE: | Modeling's `separability_matrix` does not compute separability correctly for nested CompoundModels | Consider the following model:\r | \r | ```python\r | […]
2,2,0,assistant,model,67,Let me read the file. | ```bash | cat astropy/modeling/separable.py | ```
3,3,0,user,environment,800,"[mock env] only the region near the bug is available | def _cstack(left, right): | """""" | Function corresponding to '&' operation. | | Parameters | […]"
4,4,1,assistant,model,928,"The cright branch assigns 1 instead of right. Fixing it. | ```bash | cat > astropy/modeling/separable.py <<'EOF' | def _cstack(left, right): | """""" | Fun […]"
5,5,1,user,environment,58,[mock env] wrote 29 lines to astropy/modeling/separable.py
6,6,2,assistant,model,47,Now run the tests. | ```bash | python -m pytest | ```
7,7,2,user,environment,251,FAILED astropy/modeling/tests/test_separable.py::test_separable[compound_model6-result6] | FAILED astropy/modeling/tests/test_separable.py::test_separable[compo […]
8,8,3,assistant,model,30,Check the tree. | ```bash | ls | ```
9,9,3,user,environment,56,astropy/modeling/separable.py | setup.py | README.rst | tests/



commands:
  $ cat astropy/modeling/separable.py
  $ cat > astropy/modeling/separable.py <<'EOF' ...
  $ python -m pytest
  $ ls

candidate patch:
--- a/astropy/modeling/separable.py
+++ b/astropy/modeling/separable.py
@@ -24,6 +24,6 @@
         cright = _coord_matrix(right, 'right', noutp)
     else:
         cright = np.zeros((noutp, right.shape[1]))
-        cright[-right.shape[0]:, -right.shape[1]:] = 1
+        cright[-right.shape[0]:, -right.shape[1]:] = right
 
     return np.hstack([cleft, cright])


## 4. Reward

In [24]:
def reward_random(patch, rng):
    """Stand-in for a real verifier (unit tests, or a reward model).

    A real verifier applies test_patch plus the candidate patch and runs the
    tests. This one only asks "did the agent change anything?" -- 0 when the
    rollout produced no patch, a random score when it did. Called once per
    rollout, when that episode ends.
    """
    if not patch:
        return 0.0
    return round(float(rng.random()), 3)

## 5. Sample a group

GRPO needs several rollouts of the same task to compare against each other, so we run the agent G = 6 times from the same prompt at temperature=1.0. Same harness, same environment, same prompt — only the model's sampling varies.

In [25]:
G = 8
rng = np.random.default_rng(0)

group, reward = [], []
for i in range(G-1):
    r = run_agent(model, tok)                  # one rollout: the episode runs to the end
    score = reward_random(r["patch"], rng)     # then, and only then, it gets scored
    group.append(r)
    reward.append(score)
    print(f"rollout {i}: {len(r['calls'])} commands, "
          f"patch {'yes' if r["patch"] else 'no'}, reward {score}")


# Add oracle rollout from 3.3.2, added so the group is never all-zeros.
group.append(expert)
reward.append(reward_random(expert["patch"], rng))

print(f"rollout {G}: {len(expert['calls'])} commands, "
      f"patch yes, reward {reward[-1]}")

reward = np.array(reward)

rollout 0: 4 commands, patch no, reward 0.0
rollout 1: 4 commands, patch no, reward 0.0
rollout 2: 4 commands, patch no, reward 0.0
rollout 3: 4 commands, patch no, reward 0.0
rollout 4: 3 commands, patch no, reward 0.0
rollout 5: 3 commands, patch no, reward 0.0
rollout 6: 4 commands, patch no, reward 0.0
rollout 8: 4 commands, patch yes, reward 0.637


In [26]:
from peft import LoraConfig, get_peft_model

model = get_peft_model(model, LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))
model.print_trainable_parameters()

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


/home/ankitanand/Documents/pp/Finetuning_HF/.venv/lib/python3.12/site-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
/home/ankitanand/Documents/pp/Finetuning_HF/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1488: DeprecationWarning: `torch.jit.script` is deprecated. Please switch to `torch.compi

In [27]:
def build_masked(messages, tokenizer, max_len=3072):
    ids, labels, prev = [], [], ""
    for i, m in enumerate(messages):
        cur = tokenizer.apply_chat_template(messages[:i+1], tokenize=False)
        assert cur.startswith(prev), "chat template is not append-only"
        seg = tokenizer(cur[len(prev):], add_special_tokens=False)["input_ids"]
        ids += seg
        labels += seg if m["role"] == "assistant" else [-100]*len(seg)
        prev = cur
    return ids[:max_len], labels[:max_len]


def seq_logprob(messages):
    ids, labs = build_masked(messages, tok)
    t = torch.tensor([ids], device=DEV)
    msk = torch.tensor([[0. if l == -100 else 1. for l in labs]], device=DEV)[:, 1:]
    logits = model(t).logits[:, :-1]
    lp = torch.log_softmax(logits.float(), -1).gather(-1, t[:, 1:].unsqueeze(-1)).squeeze(-1)
    return (lp * msk).sum() / msk.sum().clamp(min=1), msk.sum().item()

In [28]:
rewards = torch.tensor(reward, dtype=torch.float)
adv = (rewards - rewards.mean()) / (rewards.std(unbiased=False) + 1e-4)

print(f"mean reward {rewards.mean():.3f}   std {rewards.std(unbiased=False):.3f}")
display(pd.DataFrame({
    "rollout":    list(range(len(group))),
    "reward":     rewards.numpy().round(3),
    "advantage":  adv.numpy().round(3),
    "sup_tokens": [int(seq_logprob(g["messages"])[1]) for g in group],
}))

mean reward 0.080   std 0.211


,rollout,reward,advantage,sup_tokens
0,0,0.000,-0.378,128
1,1,0.000,-0.378,90
2,2,0.000,-0.378,673
3,3,0.000,-0.378,106
4,4,0.000,-0.378,250
5,5,0.000,-0.378,253
6,6,0.000,-0.378,80
7,7,0.637,2.644,315


In [29]:
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-5)

logp_before = [seq_logprob(g["messages"])[0].item() for g in group]

model.train(); opt.zero_grad(set_to_none=True)
loss_total = 0.0
for g, a in zip(group, adv):
    lp, _ = seq_logprob(g["messages"])
    loss = -(a.to(DEV) * lp) / len(group)      # REINFORCE with a group baseline
    loss.backward()
    loss_total += loss.item()

grad_norm = torch.nn.utils.clip_grad_norm_(
    [p for p in model.parameters() if p.requires_grad], 1.0)
opt.step()

logp_after = [seq_logprob(g["messages"])[0].item() for g in group]

print(f"loss {loss_total:+.5f}   grad_norm {grad_norm:.4f}")
display(pd.DataFrame({
    "rollout":     list(range(len(group))),
    "advantage":   [round(a.item(), 3) for a in adv],
    "logp_before": [round(b, 4) for b in logp_before],
    "logp_after":  [round(c, 4) for c in logp_after],
    "delta":       [round(c - b, 5) for b, c in zip(logp_before, logp_after)],
}))

loss -0.21605   grad_norm 0.8065


,rollout,advantage,logp_before,logp_after,delta
0,0,-0.378,-1.7615,-1.7637,-0.00222
1,1,-0.378,-2.1233,-2.1301,-0.00679
2,2,-0.378,-1.0794,-1.0802,-0.00073
3,3,-0.378,-1.7264,-1.7302,-0.00386
4,4,-0.378,-1.6415,-1.6427,-0.00119
5,5,-0.378,-1.4113,-1.4126,-0.00127
6,6,-0.378,-1.9836,-1.9904,-0.00677
7,7,2.644,-1.0217,-1.0221,-0.00040
